# Pale vs joblib / DVC — sklearn and XGBoost

3-way storage comparison for tree model checkpoint sequences.

**Methods compared:**

| Method | What it does |
|---|---|
| joblib / pickle | Naive full save — serializes the complete model every step |
| DVC (real) | File-level versioning — tracks each checkpoint file with `dvc add` + `dvc push` to a local remote |
| Pale | Tensor-level dedup — extracts per-tree arrays, hashes each, writes only new or changed tensors |

Note: safetensors is not included — it is a tensor format for neural networks and does not apply to tree models.

**Scenarios:**

1. **sklearn warm-start** — 20 checkpoints, 50 trees/step (10–1,000 trees total)
2. **XGBoost warm-start** — 20 checkpoints, 50 rounds/step (50–1,000 rounds total)
3. **sklearn hyperparameter sweep** — 5 runs from the same base, different `max_depth` / `learning_rate`
4. **XGBoost hyperparameter sweep** — 5 runs, different `eta` / `max_depth`

**DVC setup:** local remote at a temp directory. Each checkpoint tracked with `dvc add` + `dvc push`.
**Pale setup:** shared `PaleStore` root per scenario. Bytes = sum of `.chunk` files under `objects/`.
Both DVC and Pale use zstd level 3 compression.

In [ ]:
!pip install -q git+https://github.com/Olamyy/pale.git@hash-cache-no-op-path zstandard scikit-learn xgboost dvc joblib

In [ ]:
import copy
import joblib
import pickle
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import xgboost as xgb
from sklearn.ensemble import GradientBoostingClassifier

sys.path.insert(0, str(Path(".").resolve()))
from utils import pale_bytes, _fmt_bytes

from pale.store import PaleStore
from pale.adapters.sklearn import SklearnAdapter
from pale.adapters.xgboost import XGBoostAdapter

print("Imports OK")

## Shared helpers

In [ ]:
def dvc_init(repo_dir: Path, remote_dir: Path) -> None:
    repo_dir.mkdir(parents=True, exist_ok=True)
    remote_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "-q"], cwd=repo_dir, check=True)
    subprocess.run(["git", "config", "user.email", "bench@pale"], cwd=repo_dir, check=True)
    subprocess.run(["git", "config", "user.name", "Pale Bench"], cwd=repo_dir, check=True)
    subprocess.run(["dvc", "init", "-q"], cwd=repo_dir, check=True)
    subprocess.run(
        ["dvc", "remote", "add", "-d", "local", str(remote_dir)],
        cwd=repo_dir, check=True,
    )


def dvc_track_and_push(repo_dir: Path, checkpoint_path: Path) -> None:
    subprocess.run(["dvc", "add", str(checkpoint_path)], cwd=repo_dir, check=True,
                   capture_output=True)
    subprocess.run(["dvc", "push"], cwd=repo_dir, check=True, capture_output=True)


def dvc_cache_bytes(remote_dir: Path) -> int:
    return sum(p.stat().st_size for p in remote_dir.rglob("*") if p.is_file())


def raw_bytes(files) -> int:
    return sum(p.stat().st_size for p in files)


def print_comparison(
    label: str,
    n_checkpoints: int,
    raw_b: int,
    joblib_b: int,
    dvc_b: int,
    pale_b: int,
) -> None:
    def _savings(baseline, b):
        return (baseline - b) / baseline * 100 if baseline else 0

    print(f"\n{label}")
    print(f"  Checkpoints : {n_checkpoints}")
    print(f"  Raw total   : {_fmt_bytes(raw_b)}")
    print(f"  joblib      : {_fmt_bytes(joblib_b)}   (full save, no dedup)")
    print(f"  DVC         : {_fmt_bytes(dvc_b)}   (file-level dedup, {_savings(joblib_b, dvc_b):.1f}% vs joblib)")
    print(f"  Pale        : {_fmt_bytes(pale_b)}   (tensor-level dedup, {_savings(joblib_b, pale_b):.1f}% vs joblib, {_savings(dvc_b, pale_b):.1f}% vs DVC)")


print("Helpers OK")

## sklearn helpers

In [ ]:
def make_dataset(seed: int = 0):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((2000, 20)).astype(np.float32)
    y = (X[:, 0] + 0.5 * X[:, 1] > 0).astype(int)
    return X, y


def train_sklearn(
    seed: int = 0,
    n_steps: int = 20,
    trees_per_step: int = 50,
    max_depth: int = 3,
    learning_rate: float = 0.1,
):
    X, y = make_dataset(seed)
    model = GradientBoostingClassifier(
        n_estimators=trees_per_step,
        max_depth=max_depth,
        learning_rate=learning_rate,
        warm_start=True,
        random_state=seed,
    )
    models = []
    for step in range(n_steps):
        model.set_params(n_estimators=(step + 1) * trees_per_step)
        model.fit(X, y)
        models.append(copy.deepcopy(model))
    return models


print("sklearn helpers OK")

## XGBoost helpers

In [ ]:
def train_xgboost(
    seed: int = 0,
    n_steps: int = 20,
    rounds_per_step: int = 50,
    max_depth: int = 3,
    eta: float = 0.1,
):
    X, y = make_dataset(seed)
    dtrain = xgb.DMatrix(X, label=y)
    params = {
        "objective": "binary:logistic",
        "max_depth": max_depth,
        "eta": eta,
        "seed": seed,
        "verbosity": 0,
    }
    boosters = []
    booster = None
    for step in range(n_steps):
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=rounds_per_step,
            xgb_model=booster,
            verbose_eval=False,
        )
        boosters.append(booster)
    return boosters


print("XGBoost helpers OK")

---
## Scenario 1 — sklearn warm-start (20 steps, 50 trees/step)

Single run: 20 checkpoints, growing from 50 to 1,000 trees.

**Expected:** joblib stores 20 growing pickle files (full model each time).
DVC tracks those same files with file-level dedup — each file is unique
(the model grows each step), so DVC saves nothing over raw joblib.
Pale extracts per-tree arrays and only writes new trees each step.
After step 1, no existing tree ever changes — all savings come from the no-op path.

In [ ]:
print("Training sklearn warm-start (20 steps × 50 trees)...", end=" ", flush=True)
t0 = time.time()
sklearn_models = train_sklearn(seed=0)
print(f"{time.time() - t0:.1f}s")
print(f"Final model: {sklearn_models[-1].n_estimators} trees")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo   = tmp / "dvc_repo"
    dvc_remote = tmp / "dvc_remote"
    pale_root  = tmp / "pale"
    ckpt_dir   = dvc_repo / "checkpoints"   # inside the repo

    dvc_init(dvc_repo, dvc_remote)
    ckpt_dir.mkdir()

    joblib_files = []
    for step, model in enumerate(sklearn_models, 1):
        path = ckpt_dir / f"step_{step:02d}.pkl"
        joblib.dump(model, path)
        joblib_files.append(path)
        dvc_track_and_push(dvc_repo, path)

    with PaleStore(root=pale_root, run_id="sklearn_warmstart", adapter=SklearnAdapter()) as store:
        for step, model in enumerate(sklearn_models, 1):
            store.save(model, step=step)

    raw_b    = raw_bytes(joblib_files)
    joblib_b = raw_b
    dvc_b    = dvc_cache_bytes(dvc_remote)
    pale_b   = pale_bytes(pale_root)

print_comparison(
    "Scenario 1 — sklearn warm-start (20 steps × 50 trees/step)",
    len(sklearn_models), raw_b, joblib_b, dvc_b, pale_b,
)

---
## Scenario 2 — XGBoost warm-start (20 steps, 50 rounds/step)

Single run: 20 checkpoints, growing from 50 to 1,000 rounds.

**Expected:** Similar to sklearn. Each `.ubj` checkpoint grows by ~50 rounds.
DVC stores all 20 unique files. Pale writes each tree once; only the
`__skeleton__` (model metadata) changes every step.

In [ ]:
print("Training XGBoost warm-start (20 steps × 50 rounds)...", end=" ", flush=True)
t0 = time.time()
xgb_models = train_xgboost(seed=0)
print(f"{time.time() - t0:.1f}s")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo   = tmp / "dvc_repo"
    dvc_remote = tmp / "dvc_remote"
    pale_root  = tmp / "pale"
    ckpt_dir   = dvc_repo / "checkpoints"   # inside the repo

    dvc_init(dvc_repo, dvc_remote)
    ckpt_dir.mkdir()

    xgb_files = []
    for step, booster in enumerate(xgb_models, 1):
        path = ckpt_dir / f"step_{step:02d}.ubj"
        booster.save_model(str(path))
        xgb_files.append(path)
        dvc_track_and_push(dvc_repo, path)

    with PaleStore(root=pale_root, run_id="xgb_warmstart", adapter=XGBoostAdapter()) as store:
        for step, booster in enumerate(xgb_models, 1):
            store.save(booster, step=step)

    raw_b    = raw_bytes(xgb_files)
    joblib_b = raw_b
    dvc_b    = dvc_cache_bytes(dvc_remote)
    pale_b   = pale_bytes(pale_root)

print_comparison(
    "Scenario 2 — XGBoost warm-start (20 steps × 50 rounds/step)",
    len(xgb_models), raw_b, joblib_b, dvc_b, pale_b,
)

---
## Scenario 3 — sklearn hyperparameter sweep (5 runs)

5 independent runs from the same dataset (seed=0), varying `max_depth`
and `learning_rate`. Each run produces 20 checkpoints.

**Expected:** DVC stores all 5 × 20 = 100 checkpoint files independently
— no cross-run awareness. Pale shares a single store across all runs;
trees that happen to be identical across runs (same depth, same splits)
are written once. sklearn trees are deterministic given the same data and
hyperparameters, so runs with identical `max_depth` / `lr` would share
all trees — but this grid uses distinct configs, so sharing is limited.

In [ ]:
SKLEARN_HP_GRID = [
    {"run_id": "depth2_lr005",  "max_depth": 2, "learning_rate": 0.05},
    {"run_id": "depth3_lr010",  "max_depth": 3, "learning_rate": 0.10},
    {"run_id": "depth4_lr010",  "max_depth": 4, "learning_rate": 0.10},
    {"run_id": "depth3_lr020",  "max_depth": 3, "learning_rate": 0.20},
    {"run_id": "depth5_lr005",  "max_depth": 5, "learning_rate": 0.05},
]

sklearn_hp_models = {}
for cfg in SKLEARN_HP_GRID:
    t0 = time.time()
    print(f"  {cfg['run_id']}...", end=" ", flush=True)
    sklearn_hp_models[cfg["run_id"]] = train_sklearn(
        seed=0, max_depth=cfg["max_depth"], learning_rate=cfg["learning_rate"]
    )
    print(f"{time.time() - t0:.1f}s")

print(f"\nTotal checkpoints: {sum(len(v) for v in sklearn_hp_models.values())}")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo   = tmp / "dvc_repo"
    dvc_remote = tmp / "dvc_remote"
    pale_root  = tmp / "pale"
    ckpt_dir   = dvc_repo / "checkpoints"   # inside the repo

    dvc_init(dvc_repo, dvc_remote)
    ckpt_dir.mkdir()

    all_files = []
    pale_size_after = {}
    n_checkpoints = 0

    for cfg in SKLEARN_HP_GRID:
        run_id = cfg["run_id"]
        run_dir = ckpt_dir / run_id
        run_dir.mkdir()

        for step, model in enumerate(sklearn_hp_models[run_id], 1):
            path = run_dir / f"step_{step:02d}.pkl"
            joblib.dump(model, path)
            all_files.append(path)
            dvc_track_and_push(dvc_repo, path)
            n_checkpoints += 1

        with PaleStore(root=pale_root, run_id=run_id, adapter=SklearnAdapter()) as store:
            for step, model in enumerate(sklearn_hp_models[run_id], 1):
                store.save(model, step=step)

        pale_size_after[run_id] = pale_bytes(pale_root)
        print(f"  {run_id}: done")

    raw_b    = raw_bytes(all_files)
    joblib_b = raw_b
    dvc_b    = dvc_cache_bytes(dvc_remote)
    pale_b   = pale_bytes(pale_root)

print_comparison(
    "Scenario 3 — sklearn HP sweep (5 runs × 20 steps)",
    n_checkpoints, raw_b, joblib_b, dvc_b, pale_b,
)

print("\nPale store size after each run:")
prev = 0
for run_id, size in pale_size_after.items():
    print(f"  {run_id:<20} {_fmt_bytes(size):>10}  (+{_fmt_bytes(size - prev)})")
    prev = size

---
## Scenario 4 — XGBoost hyperparameter sweep (5 runs)

5 independent XGBoost runs from the same dataset, varying `eta` and `max_depth`.
Each run produces 20 checkpoints. Same shared Pale store across all runs.

**Expected:** DVC stores 100 independent `.ubj` files. Pale shares individual
tree tensors that happen to be identical across runs. Unlike sklearn, XGBoost
tree structure is less likely to produce identical tensors across different
`eta` values — cross-run savings will be modest.

In [ ]:
XGB_HP_GRID = [
    {"run_id": "depth3_eta010", "max_depth": 3, "eta": 0.10},
    {"run_id": "depth3_eta020", "max_depth": 3, "eta": 0.20},
    {"run_id": "depth4_eta010", "max_depth": 4, "eta": 0.10},
    {"run_id": "depth4_eta005", "max_depth": 4, "eta": 0.05},
    {"run_id": "depth5_eta010", "max_depth": 5, "eta": 0.10},
]

xgb_hp_models = {}
for cfg in XGB_HP_GRID:
    t0 = time.time()
    print(f"  {cfg['run_id']}...", end=" ", flush=True)
    xgb_hp_models[cfg["run_id"]] = train_xgboost(
        seed=0, max_depth=cfg["max_depth"], eta=cfg["eta"]
    )
    print(f"{time.time() - t0:.1f}s")

print(f"\nTotal checkpoints: {sum(len(v) for v in xgb_hp_models.values())}")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo   = tmp / "dvc_repo"
    dvc_remote = tmp / "dvc_remote"
    pale_root  = tmp / "pale"
    ckpt_dir   = dvc_repo / "checkpoints"   # inside the repo

    dvc_init(dvc_repo, dvc_remote)
    ckpt_dir.mkdir()

    all_files = []
    pale_size_after = {}
    n_checkpoints = 0

    for cfg in XGB_HP_GRID:
        run_id = cfg["run_id"]
        run_dir = ckpt_dir / run_id
        run_dir.mkdir()

        for step, booster in enumerate(xgb_hp_models[run_id], 1):
            path = run_dir / f"step_{step:02d}.ubj"
            booster.save_model(str(path))
            all_files.append(path)
            dvc_track_and_push(dvc_repo, path)
            n_checkpoints += 1

        with PaleStore(root=pale_root, run_id=run_id, adapter=XGBoostAdapter()) as store:
            for step, booster in enumerate(xgb_hp_models[run_id], 1):
                store.save(booster, step=step)

        pale_size_after[run_id] = pale_bytes(pale_root)
        print(f"  {run_id}: done")

    raw_b    = raw_bytes(all_files)
    joblib_b = raw_b
    dvc_b    = dvc_cache_bytes(dvc_remote)
    pale_b   = pale_bytes(pale_root)

print_comparison(
    "Scenario 4 — XGBoost HP sweep (5 runs × 20 steps)",
    n_checkpoints, raw_b, joblib_b, dvc_b, pale_b,
)

print("\nPale store size after each run:")
prev = 0
for run_id, size in pale_size_after.items():
    print(f"  {run_id:<20} {_fmt_bytes(size):>10}  (+{_fmt_bytes(size - prev)})")
    prev = size

---
## Summary

| Scenario | Checkpoints | joblib | DVC | Pale | Pale vs joblib | Pale vs DVC |
|---|---|---|---|---|---|---|
| 1 — sklearn warm-start (1 run × 20 steps) | 20 | TBD | TBD | TBD | TBD | TBD |
| 2 — XGBoost warm-start (1 run × 20 steps) | 20 | TBD | TBD | TBD | TBD | TBD |
| 3 — sklearn HP sweep (5 runs × 20 steps) | 100 | TBD | TBD | TBD | TBD | TBD |
| 4 — XGBoost HP sweep (5 runs × 20 steps) | 100 | TBD | TBD | TBD | TBD | TBD |

**Why DVC saves little over joblib for tree models:**
Each warm-start checkpoint is a unique file (the model grows each step).
DVC's file-level dedup only helps when two checkpoint files are byte-identical
— which never happens in a warm-start sequence where the model grows.
Pale operates at the tensor level: individual trees are written once and
never touched again regardless of how many more trees are added.

**Why Pale's savings are high for sklearn and moderate for XGBoost:**
sklearn stores individual tree arrays as separate tensors — once written,
they are never touched. XGBoost's `__skeleton__` (model metadata JSON)
changes on every step as round count and metadata grow, adding a small
but unavoidable write cost per step.